### GAN generator

In [ ]:
!pip install torch-fidelity

In [ ]:
import io, re, os, math
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Optional, Tuple
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from datasets import load_from_disk
from torchmetrics.image.fid import FrechetInceptionDistance


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

img_size = 64
latent_dim = 256
ch_base = 64

# Load Dataset
dataset_path = "/kaggle/input/datasets/splcher/animefacedataset/images"
print("loaded")


## Code

In [ ]:
def mostrar_resultados(resultados):
    """
    Muestra una lista de imágenes generadas.
    """
    plt.figure(figsize=(8, 2))
    for i,img in enumerate(resultados):
        plt.subplot(1, len(resultados), i+1)
        plt.imshow(img)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def calcular_fid(modelo, dataloader, fid_metric, n_samples=100, device=device):
    modelo.eval()
    fid_metric.reset()

    # imágenes reales del tamaño de nsamples
    real_batch = next(iter(dataloader)).to(device)
    if real_batch.size(0) > n_samples:
        real_batch = real_batch[:n_samples]

    # imágenes falsas
    fake_batch = modelo.generar(n_samples=n_samples, device=device, return_pil=False)
    # métricas
    real_prep = preparar_imag_indices(real_batch)
    fake_prep = preparar_imag_indices(fake_batch)
    fid_metric.update(real_prep, real=True)
    fid_metric.update(fake_prep, real=False)
    fid_val = fid_metric.compute().item()
    return fid_val

def mostrar_metricas(epocas, valores, num_epochs, label:str=""):
    plt.figure(figsize=(6, 2))
    plt.xlim(0, num_epochs)
    # Ajuste dinámico del límite Y basado en el histórico
    max_val = max(valores)
    plt.ylim(0, max_val * 1.1)
    
    plt.xlabel('Epoca')
    plt.ylabel(label)
    plt.plot(epocas, valores, color="r", marker='o', linewidth=1, label=label)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

def preparar_imag_indices(imgs):
    imgs = F.interpolate(
        imgs,
        size=(299, 299),
        mode="bilinear",
        align_corners=False)
    return imgs


class AnimeDataset(Dataset):
    def __init__(self, root_dir, image_size=img_size, augment=False, max_imgs=-1):
        self.root_dir = root_dir
        self.image_files = os.listdir(root_dir)[:max_imgs]
        self.image_size = image_size
        self.augment = augment
        
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])
        
        self.augment_fn = T.Compose([
            T.Resize((image_size, image_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        img = Image.open(img_path).convert("RGB")
        
        if self.augment:
            return self.augment_fn(img)
        return self.transform(img)

#### Crear modelo

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(32, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(32, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        h = F.silu(self.norm1(self.conv1(x)))
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)


# Self-at Block 
class SelfAttention(nn.Module):
    def __init__(self, channels, heads=8):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(channels, heads, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels*2),
            nn.SiLU(),
            nn.Linear(channels*2, channels))

    def forward(self, x):
        size = x.shape[-1]
        # Reshape para Attention: (Batch, Pixels, Channels)
        x = x.reshape(-1, self.channels, size * size).swapaxes(1, 2)
        x_ln = self.ln(x)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        # Regresar a (Batch, Channels, Height, Width)
        return attention_value.swapaxes(2, 1).reshape(-1, self.channels, size, size)


class Discriminator(nn.Module):
    def __init__(self, base_ch=ch_base):
        super().__init__()
        
        self.down1 = ResidualBlock(3, base_ch)
        self.down2 = ResidualBlock(base_ch, base_ch*2)
        self.down3 = ResidualBlock(base_ch*2, base_ch*4)
        self.down4 = ResidualBlock(base_ch*4, base_ch*8)
        self.attn = SelfAttention(base_ch*8)
        
        self.pool =nn.AvgPool2d(2)        
        self.out_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_ch*8 * 4 * 4, 1))
        
    def forward(self, x):
        h = self.down1(x)
        h = self.down2(self.pool(h))
        h = self.down3(self.pool(h))
        h = self.down4(self.pool(h))
        h = self.attn(h)
        h=self.pool(h)
        return self.out_layer(h)


class Generator(nn.Module):
    def __init__(self, latent_dim=latent_dim, base_ch=ch_base):
        super().__init__()
        # Entrada: vector de ruido z. Lo mapeamos a 4x4
        self.generator_input = nn.Linear(latent_dim, base_ch * 8 * 4 * 4)

        self.up1 = ResidualBlock(base_ch*8, base_ch*8)
        self.attn = SelfAttention(base_ch*8)
        self.up2 = ResidualBlock(base_ch*8, base_ch*4)
        self.up3 = ResidualBlock(base_ch*4, base_ch*2) 
        self.up4 = ResidualBlock(base_ch*2, base_ch)

        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")

        self.out_conv = nn.Sequential(
            ResidualBlock(base_ch, base_ch),
            nn.Conv2d(base_ch, 3, kernel_size=3, padding=1),
            nn.Tanh())

    def forward(self, z):
        h = self.generator_input(z).view(z.size(0), -1, 4, 4)
        h = self.up1(self.upsample(h))
        h = self.attn(h)
        h = self.up2(self.upsample(h))
        h = self.up3(self.upsample(h))
        h = self.up4(self.upsample(h))
        return self.out_conv(h)

    @torch.no_grad()
    def generar(self, n_samples=4, device=device, return_pil=True):
        self.eval()
        x = torch.randn(n_samples, latent_dim, device=device)
        x = self.forward(x)
        # normalizar
        x_norm = (x * 0.5 + 0.5).clamp(0, 1)

        if return_pil: return [TF.to_pil_image(img.cpu()) for img in x_norm] # convertir a PIL
        # Devolver tensores pytorch
        return x_norm


In [ ]:
# === CONFIGURACIÓN ===
# PARAMETROS DE CH_BASE Y LATENT DIM DEFINIDOS AL COMIENZO DEL CÓDIGO

batch_size = 256
lrG = 2e-4
lrD = 1e-4


In [ ]:
# Setup Datasets
train_ds = AnimeDataset(root_dir=dataset_path, image_size=img_size, augment=False)

dataloader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f"usar para entrenar {len(train_ds)} ")

#### Entrenamiento

In [ ]:
G = Generator(latent_dim=latent_dim, base_ch=ch_base).to(device)
D = Discriminator(base_ch=ch_base).to(device)

In [ ]:
optG = optim.Adam(G.parameters(), lr=lrG, betas=(0.5, 0.999))
optD = optim.Adam(D.parameters(), lr=lrD, betas=(0.5, 0.999))

criterion = nn.BCEWithLogitsLoss()

In [ ]:
# entrenamiento
num_epochs = 121

#métricas históricas
fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)

# Listas para guardar el historial
historial_fid = []
historial_epocas = []
historial_lossD = []
historial_lossG = []

for epoch in range(num_epochs):
    G.train()
    D.train()
    lossD_epoch = 0
    lossG_epoch = 0

    for imgs in dataloader:
        imgs = imgs.to(device)
        b_size = imgs.size(0) # salta error en el cálculo del error por coger tamaños distintos de batchs sino
        
        # Añadimos label smoothing para que D no sea tan agresivo
        real_labels = torch.ones(b_size, 1, device=device) * 0.9 
        fake_labels = torch.zeros(b_size, 1, device=device)
        
        # imgs falsas  (no usamos función generar pq no retropropaga gradientes)
        z = torch.randn(b_size, latent_dim, device=device)
        fake_imgs = G(z)

        # Discriminador
        optD.zero_grad()

        out_real = D(imgs)
        lossD_real = criterion(out_real, real_labels)
        out_fake = D(fake_imgs.detach())
        lossD_fake = criterion(out_fake, fake_labels)

        lossD = (lossD_real + lossD_fake) / 2
        lossD.backward()
        optD.step()

        # Generador
        optG.zero_grad()
        
        out_fake_G = D(fake_imgs)
        lossG = criterion(out_fake_G, real_labels) #tasa equivocados
        lossG.backward()
        optG.step()

        #Errores
        lossD_epoch += lossD.item()
        lossG_epoch += lossG.item()
        
        
    # =========================================================
    # LOGGING
    # =========================================================   
    if epoch % 20 == 0:
        fid_actual = calcular_fid(G, dataloader, fid)
        historial_fid.append(fid_actual)
        historial_epocas.append(epoch)
        historial_lossD.append(lossD_epoch / len(dataloader))
        historial_lossG.append(lossG_epoch / len(dataloader))
            
        print(f"Epoch {epoch}/{num_epochs} | FID: {fid_actual:.4f}")
        print(f"Generator Loss: {lossG_epoch / len(dataloader):.4f} | Discriminator Loss: {lossD_epoch / len(dataloader):.4f}")
        
        mostrar_metricas(historial_epocas, historial_fid, num_epochs, "FID")
        mostrar_metricas(historial_epocas, historial_lossD, num_epochs, "loss Discriminator")
        mostrar_metricas(historial_epocas, historial_lossG, num_epochs, "loss Generator")
        
        # Generar una imagen de ejemplo condicional 
        imgs_gen = G.generar(n_samples=4)
        mostrar_resultados(imgs_gen)

#### Guardar modelos

In [ ]:
# GUARDAR MODELOS 
import os
import torch

version = "vf1"
save_dir = f"./modelos/{version}"
os.makedirs(save_dir, exist_ok=True)

torch.save(G.state_dict(), os.path.join(save_dir, "generator.pth"))
torch.save(D.state_dict(), os.path.join(save_dir, "discriminator.pth"))

# guardamos hiperparámetros y vocabulario
metadata = {
    "latent_dim": latent_dim,
    "ch_base": ch_base}

torch.save(metadata, os.path.join(save_dir, "metadata.pth"))

print(f" Modelos y metadatos guardados en: {save_dir}")



#### resultados

In [ ]:
imgs_gen = G.generar(n_samples=4)
mostrar_resultados(imgs_gen)

In [ ]:
# resutlados 4x3
def resultados43(resultados):
    plt.figure(figsize=(10, 8))
    # Limitamos a 12 para no exceder la rejilla de 4x3
    for i, img in enumerate(resultados):
        plt.subplot(3, 4, i + 1)
        plt.imshow(img)
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    
resultados43(G.generar(n_samples=12))

#### Importar modelos

In [ ]:
import torch
import os

# Configuración
device = "cuda" if torch.cuda.is_available() else "cpu"
version = "vf1"  
save_dir = f"./modelos/{version}"

metadata = torch.load(os.path.join(save_dir, "metadata.pth"))
G = Generator(base_ch=metadata["ch_base"], latent_dim=metadata["latent_dim"]).to(device)

# === Cargar pesos ===
G.load_state_dict(torch.load(os.path.join(save_dir, "generator.pth"), map_location=device))

print(" Modelos cargados correctamente desde", save_dir)


#### validación de métricas de resultados

In [ ]:
#moderna
import gc
import torch
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from torchmetrics.functional.image import multiscale_structural_similarity_index_measure

def evaluar_modelo_generativo(model, data_loader, n=2000, batch_size=32, device=device):
    model.eval()
    
    # Inicializamos las métricas
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    is_metric = InceptionScore(normalize=True).to(device)
    lpips = LearnedPerceptualImagePatchSimilarity(net_type='vgg').to(device)
    
    ms_ssim_acumulado = 0.0
    conteo = 0
    loader_iter = iter(data_loader)

    # Procesamos en lotes para no saturar la VRAM
    for _ in range(0, n, batch_size):
        actual_n = min(batch_size, n - conteo)
        
        with torch.no_grad():
            # imágenes reales
            batch_real = next(loader_iter)
            if isinstance(batch_real, list): batch_real = batch_real[0]
            batch_real = batch_real[:actual_n].to(device).float()
            if batch_real.max() > 1.0: batch_real /= 255.0
            
            # imágenes falsas
            batch_gen = model.generar(n_samples=actual_n, device=device, return_pil=False)
            
            batch_real_proc = preparar_imag_indices(batch_real)
            batch_gen_proc = preparar_imag_indices(batch_gen)
            
            # actualizar métricas (acumulan estadísticas)
            fid.update(batch_real_proc, real=True)
            fid.update(batch_gen_proc, real=False)
            is_metric.update(batch_gen_proc)

            # Limpieza de VRAM
            torch.cuda.empty_cache()
            gc.collect()
            
            # Métricas par a par (promediamos sobre la marcha)
            lpips.update(batch_gen_proc, batch_real_proc)
            ms_ssim_acumulado += multiscale_structural_similarity_index_measure(
                batch_gen_proc, batch_real_proc, data_range=1.0)
            
            conteo += actual_n

    # Cálculo final
    is_mean, _ = is_metric.compute()
    
    return {
        "FID": fid.compute().item(),
        "IS": is_mean.item(),
        "LPIPS": lpips.compute().item(),
        "MS-SSIM": (ms_ssim_acumulado / (conteo / batch_size)).item()
    }

for i in range(10):
    print(evaluar_modelo_generativo(G, dataloader))

#### Tiempo de inferencia medio

In [ ]:
import time
import torch
import numpy as np

def medir_latencia_estadistica(model, n_iterations=1000, device=device):
    """
    Calcula la media y desviación estándar del tiempo de inferencia para generar 
    una única imagen (Batch Size = 1).
    """
    model.eval()
    tiempos = []
    
    print(f"Iniciando medición estadística sobre {n_iterations} iteraciones...")

    with torch.no_grad():
        # Fase de calentamiento (Warm-up)
        # Necesaria para estabilizar frecuencias de reloj y cargar kernels de CUDA
        for _ in range(10):
            _ = model.generar(n_samples=1, device=device, return_pil=False)
        
        if device == "cuda":
            torch.cuda.synchronize()

        # Ciclo de medición individual
        for i in range(n_iterations):
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización previa
            
            # Usamos perf_counter para obtener la mayor resolución temporal posible
            t_inicio = time.perf_counter()
            
            _ = model.generar(n_samples=1, device=device, return_pil=False)
            
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización posterior para asegurar el fin del cálculo
            
            t_fin = time.perf_counter()
            tiempos.append(t_fin - t_inicio)

    # Cálculo de métricas descriptivas
    tiempos_arr = np.array(tiempos)
    mean_time = np.mean(tiempos_arr)
    std_time = np.std(tiempos_arr)
    
    print(f"Resultados de Inferencia (n=1 por iteración)")
    print(f"mean: {mean_time:.6f} s,std: {std_time:.6f} s")

    return {
        "mean": mean_time,
        "std": std_time}

medir_latencia_estadistica(G)